
# Greedy model soup for PPO checkpoints

This notebook runs **Track 1**: greedy weight-space model soup.

It starts from one checkpoint, tries candidates one by one, and keeps a candidate **only if** the soup improves on the chosen evaluation metric.

Default behavior:

- start from `START_MODEL`
- try each checkpoint in `CANDIDATE_POOL`
- build a **trial soup** = average of current accepted members plus the candidate
- evaluate the trial soup on the configured suite
- **accept** the candidate if the metric improves
- continue until the soup is cooked and hopefully less idiotic than its ancestors

The notebook is self-contained around your current PPO/C4 stack and does not depend on the earlier distillation notebook.


In [1]:

import copy
import json
import math
import random
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from C4.CNet192 import save_cnet192
from C4.connect4_env import Connect4Env
from C4.fast_connect4_lookahead import Connect4Lookahead
from PPO.actor_critic import ActorCritic, TransferCfg, NEG_INF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda



## Config

Edit this cell first.

A good default starting point for your current HOF is:

- start with one strong generalist
- try a compact pool of 4 to 5 similarly strong generalists
- let greedy soup decide which ones actually play well together

You can keep `SORT_POOL_BY_BASELINE = False` if you want to respect the order you type in.
Set it to `True` if you want the notebook to pre-score the pool and try stronger standalone candidates first.


In [2]:

# -------------------------------
# Main soup config
# -------------------------------

START_MODEL = "PPO_Models/PPO_827.pt"

CANDIDATE_POOL = [
    "PPO_Models/PPO_817.pt",
    "PPO_Models/PPO_820.pt",
    "PPO_Models/PPO_834.pt",
    "PPO_Models/PPO_832.pt",
    "PPO_Models/PPO_836.pt",
    "PPO_Models/PPO_909.pt",
    "PPO_Models/PPO_914.pt",
]

# Optional: if True, evaluate standalone candidates first and sort pool by standalone score.
SORT_POOL_BY_BASELINE = True

# Which metric decides acceptance?
# Options created by this notebook:
#   "GS_CUSTOM"   weighted suite score
#   "AVG_SCORE"   simple mean across suite opponents
#   "LA_HARD"     mean of hard tactical opponents only (LA-3, LA-5, LA-6, LA-7, LA-9, LA-11, LA-13 if present)
METRIC_TO_MAXIMIZE = "GS_CUSTOM"

# Minimum improvement required to accept a candidate.
# Set to 0.0 for pure greedy. Small positive value avoids accepting noise.
MIN_IMPROVEMENT = 0.0

# Deterministic argmax for model actions during evaluation.
MODEL_DETERMINISTIC = True

# Reproducibility
SEED = 666

# Output name for the final soup checkpoint.
SOUP_TAG = "PPO_SOUP_TRACK1"

# Save result artifacts
SAVE_SOUP_CHECKPOINT = True
SAVE_RESULTS_XLSX = True
SAVE_RESULTS_JSON = True

# -------------------------------
# Fast suite used during greedy selection
# Keep this reasonably small, because it is run many times.
# -------------------------------

FAST_EVAL_OPPONENTS = OrderedDict({
    "Random":   {"type": "random",    "games": 60},
    "Leftmost": {"type": "leftmost",  "games": 20},
    "Center":   {"type": "center",    "games": 40},
    "LA-1":     {"type": "lookahead", "depth": 1,  "games": 20},
    "LA-2":     {"type": "lookahead", "depth": 2,  "games": 20},
    "LA-3":     {"type": "lookahead", "depth": 3,  "games": 20},
    "LA-4":     {"type": "lookahead", "depth": 4,  "games": 10},
    "LA-5":     {"type": "lookahead", "depth": 5,  "games": 10},
    "LA-6":     {"type": "lookahead", "depth": 6,  "games": 10},
    "LA-7":     {"type": "lookahead", "depth": 7,  "games": 8},
    "LA-9":     {"type": "lookahead", "depth": 9,  "games": 6},
    "LA-11":    {"type": "lookahead", "depth": 11, "games": 4},
    "LA-13":    {"type": "lookahead", "depth": 13, "games": 4},
})

# -------------------------------
# Optional final suite
# Usually same as your standard suite, or equal to fast suite if you just want a quick run.
# -------------------------------

FINAL_EVAL_OPPONENTS = OrderedDict({
    "Random":   {"type": "random",    "games": 200},
    "Leftmost": {"type": "leftmost",  "games": 100},
    "Center":   {"type": "center",    "games": 200},
    "LA-1":     {"type": "lookahead", "depth": 1,  "games": 100},
    "LA-2":     {"type": "lookahead", "depth": 2,  "games": 100},
    "LA-3":     {"type": "lookahead", "depth": 3,  "games": 100},
    "LA-4":     {"type": "lookahead", "depth": 4,  "games": 30},
    "LA-5":     {"type": "lookahead", "depth": 5,  "games": 20},
    "LA-6":     {"type": "lookahead", "depth": 6,  "games": 12},
    "LA-7":     {"type": "lookahead", "depth": 7,  "games": 10},
    "LA-9":     {"type": "lookahead", "depth": 9,  "games": 6},
    "LA-11":    {"type": "lookahead", "depth": 11, "games": 4},
    "LA-13":    {"type": "lookahead", "depth": 13, "games": 4},
})


In [3]:

# -------------------------------
# Helpers: reproducibility, paths, env
# -------------------------------

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def make_env() -> Connect4Env:
    return Connect4Env()

CENTER_ORDER = [3, 4, 2, 5, 1, 6, 0]

def center_tiebreak(indices: List[int]) -> int:
    idx_set = set(int(i) for i in indices)
    for c in CENTER_ORDER:
        if c in idx_set:
            return c
    return int(sorted(indices)[0])

def ensure_state_tensor(state: np.ndarray, device: torch.device) -> torch.Tensor:
    x = torch.as_tensor(state, dtype=torch.float32, device=device)
    if x.dim() == 2:
        x = x.unsqueeze(0).unsqueeze(0)   # (1,1,6,7)
    elif x.dim() == 3:
        x = x.unsqueeze(0)                # (1,C,6,7)
    elif x.dim() != 4:
        raise ValueError(f"Unexpected state shape: {tuple(x.shape)}")
    return x

def state_to_board_pov(state: np.ndarray) -> np.ndarray:
    s = np.asarray(state)
    if s.ndim == 4:
        if s.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got {s.shape}")
        s = s[0]
    if s.ndim == 3:
        if s.shape[0] == 1:
            s = s[0]
        else:
            raise ValueError(f"Expected single-channel POV board, got {s.shape}")
    if s.shape != (6, 7):
        raise ValueError(f"Expected board shape (6,7), got {s.shape}")
    return s.astype(np.int8, copy=False)


In [4]:

# -------------------------------
# Checkpoint loading and soup utilities
# -------------------------------

def load_actor_critic_from_ckpt(path: str | Path, device: torch.device) -> ActorCritic:
    path = Path(path)
    ac = ActorCritic.from_cnet192_checkpoint(
        path=str(path),
        device=device,
        transfer=TransferCfg(
            strict_load=True,
            freeze_conv=False,
        ),
    )
    ac.eval()
    for p in ac.parameters():
        p.requires_grad_(False)
    return ac

def get_net_state_dict(ac: ActorCritic) -> OrderedDict:
    return OrderedDict((k, v.detach().cpu().clone()) for k, v in ac.net.state_dict().items())

def assert_compatible_state_dicts(reference: OrderedDict, other: OrderedDict, label: str = "") -> None:
    ref_keys = list(reference.keys())
    oth_keys = list(other.keys())
    if ref_keys != oth_keys:
        raise ValueError(f"Incompatible keys for {label}.")
    for k in ref_keys:
        if reference[k].shape != other[k].shape:
            raise ValueError(f"Incompatible tensor shape for key '{k}' in {label}: {reference[k].shape} vs {other[k].shape}")

def average_state_dicts(state_dicts: List[OrderedDict], weights: Optional[List[float]] = None) -> OrderedDict:
    if len(state_dicts) == 0:
        raise ValueError("average_state_dicts needs at least one state dict.")

    if weights is None:
        weights = [1.0 / len(state_dicts)] * len(state_dicts)
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
        weights = weights.tolist()

    ref = state_dicts[0]
    for i, sd in enumerate(state_dicts[1:], start=1):
        assert_compatible_state_dicts(ref, sd, label=f"state_dict[{i}]")

    out = OrderedDict()
    keys = list(ref.keys())

    for k in keys:
        t0 = ref[k]
        if torch.is_floating_point(t0):
            acc = None
            for w, sd in zip(weights, state_dicts):
                term = sd[k].float() * float(w)
                acc = term if acc is None else acc + term
            out[k] = acc.to(dtype=t0.dtype)
        else:
            # Non-floating buffers should match across compatible checkpoints.
            same = True
            for sd in state_dicts[1:]:
                if not torch.equal(t0, sd[k]):
                    same = False
                    break
            if not same:
                raise ValueError(f"Non-floating tensor mismatch for key '{k}'.")
            out[k] = t0.clone()
    return out

def build_soup_model(members: List[ActorCritic], member_paths: List[str]) -> ActorCritic:
    if len(members) == 0:
        raise ValueError("Soup must contain at least one member.")
    soup = copy.deepcopy(members[0])
    avg_sd = average_state_dicts([get_net_state_dict(m) for m in members])
    soup.net.load_state_dict(avg_sd, strict=True)
    soup.eval()
    for p in soup.parameters():
        p.requires_grad_(False)
    soup._soup_member_paths = list(member_paths)
    return soup

def pretty_name(path: str | Path) -> str:
    return Path(path).stem


In [5]:

# -------------------------------
# Opponent policies
# -------------------------------

class RandomOpponent:
    def __init__(self, seed: int = 0):
        self.rng = np.random.default_rng(seed)

    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(self.rng.choice(legal_actions))

class LeftmostOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(min(legal_actions))

class CenterOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return center_tiebreak(list(legal_actions))

class LookaheadOpponent:
    def __init__(self, depth: int):
        self.depth = int(depth)
        self.la = Connect4Lookahead()
        self.la.OPENING_RANDOM = False

    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        if len(legal_actions) == 1:
            return int(legal_actions[0])

        board = state_to_board_pov(state)
        scores = np.asarray(self.la.n_step_action_scores(board, player=1, depth=self.depth), dtype=np.float64)

        mask = np.zeros(7, dtype=bool)
        mask[legal_actions] = True
        scores[~mask] = -1e18

        best = np.max(scores[mask])
        best_cols = [c for c in legal_actions if abs(scores[c] - best) <= 1e-12]
        return center_tiebreak(best_cols)

def make_opponent(cfg: Dict, seed: int = 0):
    t = cfg["type"]
    if t == "random":
        return RandomOpponent(seed=seed)
    if t == "leftmost":
        return LeftmostOpponent()
    if t == "center":
        return CenterOpponent()
    if t == "lookahead":
        return LookaheadOpponent(depth=int(cfg["depth"]))
    raise ValueError(f"Unknown opponent type: {t}")

@torch.no_grad()
def choose_model_action(model: ActorCritic, state: np.ndarray, legal_actions: List[int], deterministic: bool = True) -> int:
    x = ensure_state_tensor(state, device=device)
    logits, _ = model(x)

    legal_mask = torch.zeros_like(logits, dtype=torch.bool)
    legal_mask[0, legal_actions] = True
    masked_logits = logits.masked_fill(~legal_mask, NEG_INF)

    if deterministic:
        vals = masked_logits[0].detach().cpu().numpy()
        best = np.max(vals[legal_actions])
        best_cols = [c for c in legal_actions if abs(vals[c] - best) <= 1e-12]
        return center_tiebreak(best_cols)

    probs = torch.softmax(masked_logits, dim=-1)[0].detach().cpu().numpy()
    probs = probs / probs.sum()
    return int(np.random.choice(np.arange(7), p=probs))


In [6]:

# -------------------------------
# Evaluation
# -------------------------------

def play_one_game(model: ActorCritic, opponent, model_starts: bool, seed: int) -> int:
    '''
    Returns:
        +1 if model wins
         0 if draw
        -1 if model loses
    '''
    env = make_env()
    state = env.reset()
    model_turn = bool(model_starts)

    for _ in range(42):
        legal = env.available_actions()
        if not legal:
            return 0

        if model_turn:
            action = choose_model_action(model, state, legal, deterministic=MODEL_DETERMINISTIC)
        else:
            action = int(opponent.choose(state, legal))

        prev_model_turn = model_turn
        state, reward, done = env.step(action)

        if done:
            # Assumes reward is from the mover perspective, which matches your PPO notebooks.
            if reward > 0:
                return +1 if prev_model_turn else -1
            return 0

        model_turn = not model_turn

    return 0

def opponent_weight(label: str, base: float = 1.4) -> float:
    if label == "Random":
        return 1.0
    if label in ("Leftmost", "Center"):
        return 1.0
    m = re.search(r"(\d+)", label)
    if m is None:
        return 1.0
    depth = int(m.group(1))
    return float(base ** depth)

def summarize_suite_row(row: Dict, suite: Dict[str, Dict]) -> Dict:
    score_cols = []
    hard_cols = []

    for label in suite.keys():
        if label in row:
            score_cols.append(label)
            if label in {"LA-3", "LA-5", "LA-6", "LA-7", "LA-9", "LA-11", "LA-13"}:
                hard_cols.append(label)

    avg_score = float(np.mean([row[c] for c in score_cols])) if score_cols else 0.0
    la_hard = float(np.mean([row[c] for c in hard_cols])) if hard_cols else 0.0

    weights = np.array([opponent_weight(c) for c in score_cols], dtype=np.float64)
    vals = np.array([row[c] for c in score_cols], dtype=np.float64)
    gs_custom = float((weights * vals).sum() / weights.sum()) if len(score_cols) else 0.0

    row["AVG_SCORE"] = avg_score
    row["LA_HARD"] = la_hard
    row["GS_CUSTOM"] = gs_custom
    return row

def evaluate_model_on_suite(
    model: ActorCritic,
    suite: Dict[str, Dict],
    model_name: str,
    seed: int = 0,
    show_progress: bool = True,
) -> Tuple[pd.DataFrame, Dict]:
    row = OrderedDict()
    row["MODEL"] = model_name

    all_details = {}
    iterable = suite.items()
    if show_progress:
        iterable = tqdm(list(iterable), desc=f"Evaluating {model_name}", leave=False)

    for idx, (label, cfg) in enumerate(iterable):
        games = int(cfg["games"])
        opponent = make_opponent(cfg, seed=seed + 1000 * (idx + 1))

        wins = 0
        losses = 0
        draws = 0

        for g in range(games):
            model_starts = (g % 2 == 0)
            result = play_one_game(
                model=model,
                opponent=opponent,
                model_starts=model_starts,
                seed=seed + idx * 10000 + g,
            )
            if result > 0:
                wins += 1
            elif result < 0:
                losses += 1
            else:
                draws += 1

        score = (wins + 0.5 * draws) / games if games > 0 else 0.0
        row[label] = float(score)
        all_details[label] = {
            "wins": wins,
            "losses": losses,
            "draws": draws,
            "games": games,
            "score": float(score),
        }

    row = summarize_suite_row(row, suite)
    df = pd.DataFrame([row])
    return df, all_details

def display_eval(df: pd.DataFrame, metric: str = METRIC_TO_MAXIMIZE):
    score_cols = [c for c in df.columns if c.startswith("LA-") or c in {"Random", "Leftmost", "Center"}]
    extra_cols = ["AVG_SCORE", "LA_HARD", "GS_CUSTOM"]
    cols = ["MODEL"] + score_cols + [c for c in extra_cols if c in df.columns]
    show = df[cols].copy()
    for c in show.columns:
        if c != "MODEL":
            show[c] = show[c].map(lambda x: f"{float(x):.3f}")
    display(show)
    print(f"Metric [{metric}] =", float(df.iloc[0][metric]))


In [7]:

# -------------------------------
# Optional: preload and validate checkpoint pool
# -------------------------------

all_paths = [START_MODEL] + [p for p in CANDIDATE_POOL if p != START_MODEL]
print("Loading checkpoints...")

loaded_models: Dict[str, ActorCritic] = {}
failed_paths = {}

for p in all_paths:
    try:
        loaded_models[p] = load_actor_critic_from_ckpt(p, device=device)
        print("Loaded:", p)
    except Exception as e:
        failed_paths[p] = str(e)
        print("FAILED:", p)
        print("   ", e)

if START_MODEL not in loaded_models:
    raise RuntimeError("START_MODEL failed to load, soup cannot start.")

ref_sd = get_net_state_dict(loaded_models[START_MODEL])
compatible_pool = []
incompatible_pool = []

for p in CANDIDATE_POOL:
    if p not in loaded_models:
        incompatible_pool.append((p, "load failed"))
        continue
    try:
        assert_compatible_state_dicts(ref_sd, get_net_state_dict(loaded_models[p]), label=p)
        compatible_pool.append(p)
    except Exception as e:
        incompatible_pool.append((p, str(e)))

print("\nCompatible pool:")
for p in compatible_pool:
    print("  ", p)

if incompatible_pool:
    print("\nExcluded / incompatible candidates:")
    for p, msg in incompatible_pool:
        print("  ", p, "->", msg)

CANDIDATE_POOL = compatible_pool


Loading checkpoints...
Loaded: PPO_Models/PPO_827.pt
Loaded: PPO_Models/PPO_817.pt
Loaded: PPO_Models/PPO_820.pt
Loaded: PPO_Models/PPO_834.pt
Loaded: PPO_Models/PPO_832.pt
Loaded: PPO_Models/PPO_836.pt
Loaded: PPO_Models/PPO_909.pt
Loaded: PPO_Models/PPO_914.pt

Compatible pool:
   PPO_Models/PPO_817.pt
   PPO_Models/PPO_820.pt
   PPO_Models/PPO_834.pt
   PPO_Models/PPO_832.pt
   PPO_Models/PPO_836.pt
   PPO_Models/PPO_909.pt
   PPO_Models/PPO_914.pt



## Optional standalone prescore

If enabled, this scores each standalone candidate on the **fast suite** and sorts the trial order by that score.

This is often useful because greedy soup usually behaves better when stronger standalone models are tried earlier.


In [8]:

baseline_rows = []

# Start model baseline
start_df_fast, start_details_fast = evaluate_model_on_suite(
    model=loaded_models[START_MODEL],
    suite=FAST_EVAL_OPPONENTS,
    model_name=pretty_name(START_MODEL),
    seed=SEED,
    show_progress=True,
)
display_eval(start_df_fast)

if SORT_POOL_BY_BASELINE and len(CANDIDATE_POOL) > 0:
    print("\nScoring standalone candidates on FAST_EVAL_OPPONENTS...")
    for p in CANDIDATE_POOL:
        df_i, _ = evaluate_model_on_suite(
            model=loaded_models[p],
            suite=FAST_EVAL_OPPONENTS,
            model_name=pretty_name(p),
            seed=SEED + 1,
            show_progress=False,
        )
        row = df_i.iloc[0].to_dict()
        row["PATH"] = p
        baseline_rows.append(row)

    baseline_df = pd.DataFrame(baseline_rows).sort_values(
        by=METRIC_TO_MAXIMIZE, ascending=False
    ).reset_index(drop=True)

    show_cols = ["MODEL", METRIC_TO_MAXIMIZE, "AVG_SCORE", "LA_HARD", "GS_CUSTOM"]
    show_cols = [c for c in show_cols if c in baseline_df.columns]
    display(baseline_df[show_cols])

    CANDIDATE_POOL = baseline_df["PATH"].tolist()
    print("\nPool order after standalone prescore:")
    for p in CANDIDATE_POOL:
        print("  ", p)
else:
    print("\nPool order will be used exactly as written in CANDIDATE_POOL.")


Evaluating PPO_827:   0%|          | 0/13 [00:00<?, ?it/s]

,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_827,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.000,0.500,0.500,0.500,0.000,0.692,0.500,0.301


Metric [GS_CUSTOM] = 0.30065561765637666

Scoring standalone candidates on FAST_EVAL_OPPONENTS...


,MODEL,GS_CUSTOM,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_909,0.935977,0.921795,0.928571,0.935977
1,PPO_817,0.690387,0.844872,0.785714,0.690387
2,PPO_834,0.565180,0.769231,0.714286,0.565180
3,PPO_820,0.554663,0.767949,0.642857,0.554663
4,PPO_832,0.481472,0.535897,0.428571,0.481472
5,PPO_914,0.340014,0.692308,0.571429,0.340014
6,PPO_836,0.298851,0.615385,0.500000,0.298851



Pool order after standalone prescore:
   PPO_Models/PPO_909.pt
   PPO_Models/PPO_817.pt
   PPO_Models/PPO_834.pt
   PPO_Models/PPO_820.pt
   PPO_Models/PPO_832.pt
   PPO_Models/PPO_914.pt
   PPO_Models/PPO_836.pt



## Greedy soup run

This is the main cell.

Logic:

- current soup starts as `START_MODEL`
- for each candidate:
  - build trial soup = average(current members + candidate)
  - evaluate trial soup on the fast suite
  - if chosen metric improves by at least `MIN_IMPROVEMENT`, accept it
  - otherwise reject it
- continue until all candidates are tested


In [9]:
history_rows = []
accepted_paths = [START_MODEL]
accepted_models = [loaded_models[START_MODEL]]
current_soup = build_soup_model(accepted_models, accepted_paths)

current_fast_df, current_fast_details = evaluate_model_on_suite(
    model=current_soup,
    suite=FAST_EVAL_OPPONENTS,
    model_name="SOUP_START",
    seed=SEED,
    show_progress=True,
    tqdm_position=1,
    tqdm_leave=False,
)
current_metric = float(current_fast_df.iloc[0][METRIC_TO_MAXIMIZE])

history_rows.append({
    "step": 0,
    "candidate": pretty_name(START_MODEL),
    "accepted": True,
    "reason": "initial soup",
    "members": " + ".join(pretty_name(p) for p in accepted_paths),
    "metric_before": np.nan,
    "metric_trial": current_metric,
    "metric_after": current_metric,
})

print("\nInitial soup members:", [pretty_name(p) for p in accepted_paths])
print(f"Initial {METRIC_TO_MAXIMIZE}: {current_metric:.6f}")

outer_bar = tqdm(
    list(enumerate(CANDIDATE_POOL, start=1)),
    desc="Greedy soup candidates",
    total=len(CANDIDATE_POOL),
    position=0,
    leave=True,
)

for step_idx, candidate_path in outer_bar:
    candidate_name = pretty_name(candidate_path)
    metric_before = current_metric

    outer_bar.set_postfix({
        "candidate": candidate_name,
        "current": f"{current_metric:.4f}",
        "members": len(accepted_paths),
    })

    print("\n" + "=" * 88)
    print(f"Step {step_idx}: trying candidate {candidate_name}")

    trial_paths = accepted_paths + [candidate_path]
    trial_models = accepted_models + [loaded_models[candidate_path]]
    trial_soup = build_soup_model(trial_models, trial_paths)

    trial_df, trial_details = evaluate_model_on_suite(
        model=trial_soup,
        suite=FAST_EVAL_OPPONENTS,
        model_name="TRIAL_" + candidate_name,
        seed=SEED + step_idx,
        show_progress=True,
        tqdm_position=1,
        tqdm_leave=False,
    )

    trial_metric = float(trial_df.iloc[0][METRIC_TO_MAXIMIZE])
    delta = trial_metric - metric_before
    accepted = bool(delta >= MIN_IMPROVEMENT)

    if accepted:
        accepted_paths = trial_paths
        accepted_models = trial_models
        current_soup = trial_soup
        current_fast_df = trial_df
        current_fast_details = trial_details
        current_metric = trial_metric
        reason = f"accepted, Δ={delta:.6f}"
        print(f"ACCEPTED {candidate_name} | {METRIC_TO_MAXIMIZE}: {trial_metric:.6f} | delta={delta:.6f}")
    else:
        reason = f"rejected, Δ={delta:.6f}"
        print(f"REJECTED {candidate_name} | trial={trial_metric:.6f} | current={metric_before:.6f} | delta={delta:.6f}")

    history_rows.append({
        "step": step_idx,
        "candidate": candidate_name,
        "accepted": accepted,
        "reason": reason,
        "members": " + ".join(pretty_name(p) for p in accepted_paths),
        "metric_before": metric_before,
        "metric_trial": trial_metric,
        "metric_after": current_metric,
    })

    outer_bar.set_postfix({
        "candidate": candidate_name,
        "trial": f"{trial_metric:.4f}",
        "current": f"{current_metric:.4f}",
        "accepted": accepted,
        "members": len(accepted_paths),
    })

history_df = pd.DataFrame(history_rows)
display(history_df)

print("\nFinal accepted soup members:")
for p in accepted_paths:
    print("  ", pretty_name(p))

print(f"\nFinal FAST {METRIC_TO_MAXIMIZE}: {current_metric:.6f}")
display_eval(current_fast_df)

Evaluating SOUP_START:   0%|          | 0/13 [00:00<?, ?it/s]


Initial soup members: ['PPO_827']
Initial GS_CUSTOM: 0.300656


Greedy soup candidates:   0%|          | 0/7 [00:00<?, ?it/s]


Step 1: trying candidate PPO_909


Evaluating TRIAL_PPO_909:   0%|          | 0/13 [00:00<?, ?it/s]

...

REJECTED PPO_836 | trial=0.440313 | current=0.669202 | delta=-0.228889


,step,candidate,accepted,reason,members,metric_before,metric_trial,metric_after
0,0,PPO_827,True,initial soup,PPO_827,NaN,0.300656,0.300656
1,1,PPO_909,False,"rejected, Δ=-0.290105",PPO_827,0.300656,0.010551,0.300656
2,2,PPO_817,True,"accepted, Δ=0.029791",PPO_827 + PPO_817,0.300656,0.330446,0.330446
3,3,PPO_834,True,"accepted, Δ=0.338755",PPO_827 + PPO_817 + PPO_834,0.330446,0.669202,0.669202
4,4,PPO_820,False,"rejected, Δ=-0.228889",PPO_827 + PPO_817 + PPO_834,0.669202,0.440313,0.669202
5,5,PPO_832,False,"rejected, Δ=-0.114444",PPO_827 + PPO_817 + PPO_834,0.669202,0.554757,0.669202
6,6,PPO_914,False,"rejected, Δ=-0.655165",PPO_827 + PPO_817 + PPO_834,0.669202,0.014036,0.669202
7,7,PPO_836,False,"rejected, Δ=-0.228889",PPO_827 + PPO_817 + PPO_834,0.669202,0.440313,0.669202



Final accepted soup members:
   PPO_827
   PPO_817
   PPO_834

Final FAST GS_CUSTOM: 0.669202


,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,TRIAL_PPO_834,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.000,1.000,0.500,1.000,0.500,0.808,0.714,0.669


Metric [GS_CUSTOM] = 0.6692016843215427



## Final evaluation on the larger suite


In [10]:

final_name = SOUP_TAG + "__" + "__".join(pretty_name(p) for p in accepted_paths)

final_df, final_details = evaluate_model_on_suite(
    model=current_soup,
    suite=FINAL_EVAL_OPPONENTS,
    model_name=final_name,
    seed=SEED + 999,
    show_progress=True,
)

display_eval(final_df)


Evaluating PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834:   0%|          | 0/13 [00:00<?, ?it/s]

,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.000,1.000,0.500,1.000,0.500,0.808,0.714,0.669


Metric [GS_CUSTOM] = 0.6692016843215427


In [11]:

# -------------------------------
# Save outputs
# -------------------------------

tag_suffix = "__".join(pretty_name(p) for p in accepted_paths)
safe_tag = f"{SOUP_TAG}__{tag_suffix}"

artifacts = {}

if SAVE_SOUP_CHECKPOINT:
    out_ckpt = Path("PPO_Models") / f"{safe_tag}.pt"
    out_ckpt.parent.mkdir(parents=True, exist_ok=True)

    save_cnet192(
        model=current_soup.net,
        path=out_ckpt,
        tag=safe_tag,
        session=safe_tag,
        episode=0,
        seed=SEED,
        t=0,
        cfg_override={"use_mid_3x3": bool(getattr(current_soup.net, "use_mid_3x3", True))},
    )
    artifacts["checkpoint"] = str(out_ckpt)
    print("Saved soup checkpoint to:", out_ckpt)

if SAVE_RESULTS_XLSX:
    out_xlsx = Path(f"{safe_tag}_results.xlsx")
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        history_df.to_excel(writer, sheet_name="greedy_history", index=False)
        current_fast_df.to_excel(writer, sheet_name="fast_eval_final", index=False)
        final_df.to_excel(writer, sheet_name="final_eval", index=False)
    artifacts["xlsx"] = str(out_xlsx)
    print("Saved results workbook to:", out_xlsx)

if SAVE_RESULTS_JSON:
    out_json = Path(f"{safe_tag}_recipe.json")
    payload = {
        "tag": safe_tag,
        "start_model": START_MODEL,
        "candidate_pool": CANDIDATE_POOL,
        "accepted_paths": accepted_paths,
        "metric_to_maximize": METRIC_TO_MAXIMIZE,
        "min_improvement": MIN_IMPROVEMENT,
        "fast_metric_final": float(current_fast_df.iloc[0][METRIC_TO_MAXIMIZE]),
        "final_metric": float(final_df.iloc[0][METRIC_TO_MAXIMIZE]),
        "fast_eval_row": current_fast_df.iloc[0].to_dict(),
        "final_eval_row": final_df.iloc[0].to_dict(),
        "history": history_df.to_dict(orient="records"),
    }
    out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    artifacts["json"] = str(out_json)
    print("Saved recipe JSON to:", out_json)

print("\nArtifacts:")
for k, v in artifacts.items():
    print(f"  {k}: {v}")


Saved soup checkpoint to: PPO_Models\PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834.pt
Saved results workbook to: PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834_results.xlsx
Saved recipe JSON to: PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834_recipe.json

Artifacts:
  checkpoint: PPO_Models\PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834.pt
  xlsx: PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834_results.xlsx
  json: PPO_SOUP_TRACK1__PPO_827__PPO_817__PPO_834_recipe.json



## Notes

A few practical points:

- This notebook implements the exact greedy logic you described: **try candidate, keep only if improved, move on**.
- The soup is a **weight-space average** of accepted members.
- The final checkpoint is a single normal PPO/CNet192 checkpoint, not an inference ensemble.
- If the result is disappointing, that is useful information. It means the selected checkpoints do not live in a friendly shared basin, or your validation suite is not stressing the right blind spots.
- If this works, the accepted member list becomes the best teacher pool for **Track 2** routed distillation.
